# RAG over the FMLA Employer's Guide: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *The Employer's Guide to the Family and Medical Leave Act* (U.S. Department of Labor, 76 pages), `data/GOVPUB-L-PURL-gpo84393.pdf`

**Two pipelines over the same chunks:**
- **Pipeline 1 (baseline):** bi-encoder retrieves the top 10 chunks, the top 3 go to the LLM, giving Answer 1.
- **Pipeline 2 (re-ranking):** the same top 10 are re-scored by a cross-encoder, the new top 3 go to the LLM with the same prompt, giving Answer 2.

**Notebook layout:**
1. Setup
2. Loading and cleaning the PDF
3. Chunking
4. Embeddings and vector store
5. Pipeline 1
6. Pipeline 2
7. Failure-mode analysis
8. Latency
9. Scaling

## Step 1: Setup

Install the packages, import them, and check that the Gemini API key works.

- The retrieval models (bi-encoder and cross-encoder) run locally on the CPU.
- Only the answer generation calls the Gemini API.

In [1]:
# Install packages into the same environment this notebook runs in (%pip, not !pip).
# torch comes from PyTorch's CPU-only index: this laptop has no GPU, and the default
# CUDA build is several GB (the first install attempt timed out downloading it).
%pip install -q --no-cache-dir --default-timeout=120 torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q --no-cache-dir --default-timeout=120 -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import platform

import numpy as np
import pandas as pd
import torch
import pymupdf                 # PyMuPDF: extracts text from the PDF
import faiss                   # vector index for similarity search
import sentence_transformers
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Record the environment so the results can be reproduced later
print("python               ", platform.python_version())
print("torch                ", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("CPU threads used     ", torch.get_num_threads())
print("pymupdf              ", pymupdf.__version__)
print("faiss                ", faiss.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("numpy / pandas       ", np.__version__, "/", pd.__version__)

python                3.12.3
torch                 2.14.1+cpu | CUDA available: False
CPU threads used      10
pymupdf               1.28.2
faiss                 1.15.1
sentence-transformers 6.1.0
numpy / pandas        2.4.3 / 3.0.1


In [3]:
# Paths used throughout the notebook
DATA_PDF = "data/GOVPUB-L-PURL-gpo84393.pdf"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(DATA_PDF), "PDF not found - run the notebook from the repo root"
print("PDF size: %.1f MB" % (os.path.getsize(DATA_PDF) / 1e6))

PDF size: 5.2 MB


In [4]:
# The API key lives in .env (never committed; see .gitignore).
# load_dotenv() copies the lines of .env into environment variables.
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY missing - add it to a .env file in the repo root"

client = genai.Client(api_key=api_key)

# Model names change over time, so list the Flash models this key can call
# (speech "tts" and "image" models are skipped: we only need text answers)
flash_models = [
    m.name for m in client.models.list()
    if "flash" in m.name and "generateContent" in (m.supported_actions or [])
    and "tts" not in m.name and "image" not in m.name
]
print(len(flash_models), "Flash models available:")
for name in flash_models:
    print("  ", name)

14 Flash models available:
   models/gemini-2.5-flash
   models/gemini-flash-latest
   models/gemini-flash-lite-latest
   models/gemini-2.5-flash-lite
   models/gemini-3-flash-preview
   models/gemini-3.1-flash-lite-preview
   models/gemini-3.1-flash-lite
   models/gemini-3.5-flash
   models/gemini-3.5-flash-lite
   models/gemini-omni-flash-preview
   models/gemini-omni-1.1-flash
   models/gemini-3.6-flash
   models/gemini-3.7-flash
   models/gemini-3.8-flash


In [5]:
import time

# One model for the whole notebook, so both pipelines use exactly the same LLM.
# (gemini-2.5-flash is no longer offered to new API keys.)
GEMINI_MODEL = "gemini-3.8-flash"
assert "models/" + GEMINI_MODEL in flash_models, "model not available for this key"

# Smoke test: one tiny request to confirm the key and model work
start = time.perf_counter()
reply = client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Say hello in five words.",
    config=types.GenerateContentConfig(temperature=0),
)
elapsed = time.perf_counter() - start

usage = reply.usage_metadata
print("model          :", GEMINI_MODEL)
print("reply          :", reply.text.strip())
print("time           : %.1f s" % elapsed)
print("prompt tokens  :", usage.prompt_token_count)
print("answer tokens  :", usage.candidates_token_count)
print("thinking tokens:", usage.thoughts_token_count)

model          : gemini-3.8-flash
reply          : Hello there, how are you?
time           : 4.2 s
prompt tokens  : 7
answer tokens  : 7
thinking tokens: 269
